# GroundCheck Calibration Analysis

This notebook provides an interactive analysis of GroundCheck's confidence calibration.

**Calibration concept:** A well-calibrated system is correct ~80% of the time when it predicts 80% confidence. The Expected Calibration Error (ECE) measures the average gap between predicted confidence and actual accuracy across equal-width bins.

**ECE formula:**
$$\text{ECE} = \sum_{b=1}^{B} \frac{|B_b|}{n} \cdot |\text{conf}(B_b) - \text{acc}(B_b)|$$

**Acceptance targets (ticket 6.8):**
| Metric | Target |
|--------|--------|
| ECE | < 0.15 |
| HIGH tier accuracy | ≥ 80% |
| LOW tier accuracy | < 50% |
| Tier separation | ≥ 30 percentage points |

In [ ]:
# ── Setup ───────────────────────────────────────────────────────────────────
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Add backend root to path so we can import calibration_analysis
BACKEND_DIR = Path(".").resolve()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from calibration_analysis import (
    CalibrationRunner,
    ECECalculator,
    CalibrationPlotter,
    DEFAULT_DATASET_PATH,
    DEFAULT_OUTPUT_DIR,
    SEMANTIC_AVAILABLE,
    N_BINS,
    ECE_TARGET,
    HIGH_TIER_ACC_TARGET,
    LOW_TIER_ACC_TARGET,
    SEPARATION_TARGET,
)

# ── Configuration ───────────────────────────────────────────────────────────
API_URL      = "http://localhost:8000"
DATASET_PATH = DEFAULT_DATASET_PATH
OUTPUT_DIR   = DEFAULT_OUTPUT_DIR
MAX_Q        = None   # Set to an int (e.g. 10) for a quick smoke test
TOP_K        = 5
THRESHOLD    = 0.70   # similarity threshold to count an answer as correct

print(f"Dataset     : {DATASET_PATH}")
print(f"API URL     : {API_URL}")
print(f"Similarity  : {'semantic (sentence-transformers)' if SEMANTIC_AVAILABLE else 'token overlap F1'}")

## 1. Load Validation Dataset

In [ ]:
pairs = CalibrationRunner.load_dataset(DATASET_PATH, max_questions=MAX_Q)

# Dataset summary
from collections import Counter
type_counts = Counter(p.question_type for p in pairs)
tier_counts = Counter(p.expected_confidence_tier for p in pairs)
diff_counts = Counter(p.difficulty for p in pairs)

print(f"Total questions : {len(pairs)}")
print()
print("By question type:")
for t, c in sorted(type_counts.items()):
    print(f"  {t:<22s}  {c:3d}  ({c/len(pairs):.0%})")
print()
print("By expected tier:")
for tier, c in sorted(tier_counts.items()):
    print(f"  {tier:<10s}  {c:3d}  ({c/len(pairs):.0%})")
print()
print("By difficulty:")
for d, c in sorted(diff_counts.items()):
    print(f"  {d:<10s}  {c:3d}  ({c/len(pairs):.0%})")

In [ ]:
# Visualise dataset distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (counts, title) in zip(axes, [
    (type_counts, "Question Type"),
    (tier_counts, "Expected Tier"),
    (diff_counts, "Difficulty"),
]):
    labels = list(counts.keys())
    values = [counts[l] for l in labels]
    colors = plt.cm.Set2(range(len(labels)))
    ax.bar(labels, values, color=colors, edgecolor="black", alpha=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel("Count")
    for i, v in enumerate(values):
        ax.text(i, v + 0.2, str(v), ha="center", fontsize=9)
    ax.tick_params(axis="x", rotation=15)

fig.suptitle("Validation Dataset Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Run Calibration (submit all questions to GroundCheck)

> **Note:** This cell calls the live API. Make sure the backend is running:
> ```bash
> cd backend && uvicorn main:app --reload
> ```
>
> With 60 questions and ~5–30 s per LLM call, expect 5–30 minutes total runtime.

In [ ]:
runner = CalibrationRunner(
    api_url=API_URL,
    top_k=TOP_K,
    delay_s=0.5,
    threshold=THRESHOLD,
    output_dir=OUTPUT_DIR,
)

# Health check before starting
if not runner.client.health_check():
    print("ERROR: API not reachable. Start uvicorn first.")
else:
    records = runner.collect_predictions(pairs)
    print(f"\nCollected {len(records)} predictions.")

### (Alternative) Load existing results from a previous run

In [ ]:
# Uncomment to reload from a previously saved report instead of re-running:
#
# from calibration_analysis import PredictionRecord
# from dataclasses import fields as dc_fields
#
# report_path = OUTPUT_DIR / "calibration_report.json"
# with open(report_path) as f:
#     saved = json.load(f)
# field_names = {f.name for f in dc_fields(PredictionRecord)}
# records = [
#     PredictionRecord(**{k: v for k, v in p.items() if k in field_names})
#     for p in saved["predictions"]
# ]
# print(f"Loaded {len(records)} predictions from {report_path}")

## 3. ECE Calculation

In [ ]:
metrics = runner.compute_metrics(records)

print(f"ECE  = {metrics.ece:.4f}  (target < {ECE_TARGET})  "
      f"{'PASS ✓' if metrics.ece_pass else 'FAIL ✗'}")
print(f"Overall accuracy  = {metrics.overall_accuracy:.1%}")
print(f"Tier prediction accuracy = {metrics.tier_prediction_accuracy:.1%}")

print("\nBin breakdown:")
print(f"  {'Bin':>8}  {'n':>5}  {'Avg conf':>9}  {'Accuracy':>9}  {'Gap':>7}")
for b in metrics.bin_stats:
    if b.count > 0:
        print(f"  {b.label:>8}  {b.count:>5d}  "
              f"{b.avg_confidence:>8.1%}  {b.accuracy:>8.1%}  "
              f"{b.calibration_gap:>6.4f}")

## 4. Calibration Plot

In [ ]:
# Generate and display the reliability diagram + tier accuracy chart
plot_path = OUTPUT_DIR / "calibration_plot.png"
CalibrationPlotter.plot(metrics, plot_path)

from IPython.display import Image
Image(str(plot_path))

## 5. Tier-Specific Accuracy

In [ ]:
high_ok = (metrics.high_tier.accuracy is None or
           metrics.high_tier.accuracy >= HIGH_TIER_ACC_TARGET)
low_ok  = (metrics.low_tier.accuracy  is None or
           metrics.low_tier.accuracy  <  LOW_TIER_ACC_TARGET)

fmt = lambda v: f"{v:.1%}" if v is not None else "N/A"

print(f"HIGH   tier  (n={metrics.high_tier.count:3d}) : "
      f"{fmt(metrics.high_tier.accuracy):>7}  target ≥{HIGH_TIER_ACC_TARGET:.0%}  "
      f"{'PASS ✓' if high_ok else 'FAIL ✗'}")

print(f"MEDIUM tier  (n={metrics.medium_tier.count:3d}) : "
      f"{fmt(metrics.medium_tier.accuracy):>7}  (no target)")

print(f"LOW    tier  (n={metrics.low_tier.count:3d}) : "
      f"{fmt(metrics.low_tier.accuracy):>7}  target <{LOW_TIER_ACC_TARGET:.0%}   "
      f"{'PASS ✓' if low_ok else 'FAIL ✗'}")

if metrics.tier_separation is not None:
    print(f"\nTier separation : {metrics.tier_separation:+.1f} pp  "
          f"target ≥{SEPARATION_TARGET:.0f} pp  "
          f"{'PASS ✓' if metrics.separation_pass else 'FAIL ✗'}")

## 6. Per-Question-Type Breakdown

In [ ]:
from collections import defaultdict

type_records = defaultdict(list)
for r in records:
    type_records[r.question_type].append(r)

print(f"{'Question type':<22}  {'n':>4}  {'Correct':>7}  {'Accuracy':>9}  "
      f"{'Avg conf':>9}  {'Tier acc':>9}")
print("-" * 70)

for qtype in ["factual_lookup", "multi_step", "out_of_scope", "edge_case"]:
    recs = type_records.get(qtype, [])
    if not recs:
        continue
    n_correct  = sum(int(r.is_correct)    for r in recs)
    avg_conf   = np.mean([r.confidence_score for r in recs])
    tier_acc   = np.mean([int(r.tier_correct) for r in recs])
    print(f"  {qtype:<20}  {len(recs):>4d}  {n_correct:>7d}  "
          f"{n_correct/len(recs):>8.1%}  {avg_conf:>8.1f}  {tier_acc:>8.1%}")

In [ ]:
# Visualise per-type accuracy vs average confidence
fig, ax = plt.subplots(figsize=(9, 5))

type_labels, type_accs, type_confs, type_ns = [], [], [], []
for qtype in ["factual_lookup", "multi_step", "edge_case", "out_of_scope"]:
    recs = type_records.get(qtype, [])
    if recs:
        type_labels.append(qtype.replace("_", "\n"))
        type_accs.append(np.mean([int(r.is_correct) for r in recs]))
        type_confs.append(np.mean([r.confidence_score / 100 for r in recs]))
        type_ns.append(len(recs))

x = np.arange(len(type_labels))
width = 0.35
b1 = ax.bar(x - width/2, type_accs,  width, label="Answer accuracy", color="#4C72B0", alpha=0.8, edgecolor="black")
b2 = ax.bar(x + width/2, type_confs, width, label="Avg confidence",  color="#DD8452", alpha=0.8, edgecolor="black")

for bar, val in zip(list(b1) + list(b2), type_accs + type_confs):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f"{val:.1%}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(type_labels, fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score (fraction)")
ax.set_title("Answer Accuracy vs Avg Confidence by Question Type", fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "type_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Per-Difficulty Breakdown

In [ ]:
diff_records = defaultdict(list)
for r in records:
    diff_records[r.difficulty].append(r)

print(f"{'Difficulty':<12}  {'n':>4}  {'Accuracy':>9}  {'Avg conf':>9}")
print("-" * 42)
for diff in ["easy", "medium", "hard"]:
    recs = diff_records.get(diff, [])
    if recs:
        acc  = np.mean([int(r.is_correct) for r in recs])
        conf = np.mean([r.confidence_score for r in recs])
        print(f"  {diff:<10}  {len(recs):>4d}  {acc:>8.1%}  {conf:>8.1f}")

## 8. Confidence Score Distribution

In [ ]:
scores  = [r.confidence_score for r in records]
correct = [r.confidence_score for r in records if r.is_correct]
wrong   = [r.confidence_score for r in records if not r.is_correct]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of predicted confidence scores
ax = axes[0]
bins = np.arange(0, 110, 10)
ax.hist(correct, bins=bins, alpha=0.7, color="#2ca02c", label="Correct", edgecolor="black")
ax.hist(wrong,   bins=bins, alpha=0.7, color="#d62728", label="Incorrect", edgecolor="black")
ax.axvline(np.mean(scores), color="navy", linewidth=2, linestyle="--",
           label=f"Mean = {np.mean(scores):.1f}")
ax.set_xlabel("Predicted confidence score (0–100)")
ax.set_ylabel("Count")
ax.set_title("Confidence Score Distribution by Correctness")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Right: accuracy vs similarity score scatter
ax2 = axes[1]
sims   = [r.similarity_score for r in records]
confs  = [r.confidence_score / 100 for r in records]
colors = ["#2ca02c" if r.is_correct else "#d62728" for r in records]
ax2.scatter(sims, confs, c=colors, alpha=0.6, edgecolors="white", linewidths=0.5, s=50)
ax2.axvline(0.70, color="black", linewidth=1.5, linestyle="--", label="Correct threshold")
ax2.set_xlabel("Answer similarity score")
ax2.set_ylabel("Predicted confidence (fraction)")
ax2.set_title("Confidence vs Answer Similarity")
correct_patch = mpatches.Patch(color="#2ca02c", alpha=0.7, label="Correct")
wrong_patch   = mpatches.Patch(color="#d62728", alpha=0.7, label="Incorrect")
ax2.legend(handles=[correct_patch, wrong_patch,
           mpatches.Patch(color="none", label="── Threshold")])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "score_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Tier Prediction Confusion Matrix

In [ ]:
from collections import Counter

tiers = ["HIGH", "MEDIUM", "LOW"]
conf_matrix = np.zeros((3, 3), dtype=int)
tier_idx = {t: i for i, t in enumerate(tiers)}

for r in records:
    expected = r.expected_tier
    actual   = r.confidence_tier
    if expected in tier_idx and actual in tier_idx:
        conf_matrix[tier_idx[expected]][tier_idx[actual]] += 1

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(conf_matrix, cmap="Blues")
plt.colorbar(im, ax=ax)
ax.set_xticks(range(3)); ax.set_xticklabels(tiers)
ax.set_yticks(range(3)); ax.set_yticklabels(tiers)
ax.set_xlabel("Predicted tier")
ax.set_ylabel("Expected tier")
ax.set_title("Tier Prediction Confusion Matrix")

for i in range(3):
    for j in range(3):
        val = conf_matrix[i, j]
        color = "white" if val > conf_matrix.max() * 0.6 else "black"
        ax.text(j, i, str(val), ha="center", va="center",
                fontsize=14, fontweight="bold", color=color)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tier_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

# Overall tier accuracy
correct_tiers = sum(conf_matrix[i, i] for i in range(3))
print(f"Tier prediction accuracy: {correct_tiers}/{len(records)} = {correct_tiers/len(records):.1%}")

## 10. Per-Record Detail Table

In [ ]:
# Show all predictions sorted by similarity score (worst first)
sorted_records = sorted(records, key=lambda r: r.similarity_score)

print(f"{'ID':<6}  {'Type':<16}  {'Score':>5}  {'Tier':<7}  "
      f"{'ExpTier':<8}  {'Sim':>6}  {'Correct':<7}  {'TierOK'}")
print("-" * 78)
for r in sorted_records:
    print(
        f"  {r.qa_id:<4}  {r.question_type:<16}  {r.confidence_score:>5d}  "
        f"{r.confidence_tier:<7}  {r.expected_tier:<8}  {r.similarity_score:>5.3f}  "
        f"{'✓' if r.is_correct else '✗':<7}  {'✓' if r.tier_correct else '✗'}"
    )

## 11. Full Metrics Summary + Save Outputs

In [ ]:
# Save report and summary
runner.save_report(
    metrics,
    records,
    OUTPUT_DIR / "calibration_report.json",
)
runner.print_and_save_summary(
    metrics,
    OUTPUT_DIR / "calibration_summary.txt",
)

# Print final pass/fail table
high_ok = (metrics.high_tier.accuracy is None or
           metrics.high_tier.accuracy >= HIGH_TIER_ACC_TARGET)
low_ok  = (metrics.low_tier.accuracy  is None or
           metrics.low_tier.accuracy  <  LOW_TIER_ACC_TARGET)

print("\nAcceptance criteria pass/fail:")
print(f"  ECE < {ECE_TARGET}                          : {'PASS ✓' if metrics.ece_pass else 'FAIL ✗'}  (ECE={metrics.ece:.4f})")
print(f"  HIGH tier accuracy ≥ {HIGH_TIER_ACC_TARGET:.0%}           : {'PASS ✓' if high_ok else 'FAIL ✗'}  ({metrics.high_tier.accuracy:.1%} if metrics.high_tier.accuracy else 'N/A'})")
print(f"  LOW  tier accuracy < {LOW_TIER_ACC_TARGET:.0%}           : {'PASS ✓' if low_ok else 'FAIL ✗'}  ({metrics.low_tier.accuracy:.1%} if metrics.low_tier.accuracy else 'N/A'})")
print(f"  Tier separation ≥ {SEPARATION_TARGET:.0f} pp              : {'PASS ✓' if metrics.separation_pass else 'FAIL ✗'}  ({metrics.tier_separation:+.1f} pp if metrics.tier_separation else 'N/A'})")
print(f"\n  OVERALL  :  {'PASS ✓' if metrics.passes_all() else 'FAIL ✗'}")

print(f"\nOutput files written to: {OUTPUT_DIR}")